In [1]:
import sys
import pandas as pd
sys.path.append('../src')
from battle_engine import BattlePokemon, BattleState, TurnEngine
from synergy_scorer import TeamSynergyScorer
from ai_agent import SmartVGCAgent

In [2]:
df_pokemon = pd.read_csv('../data/processed/clustered_pokemon_data.csv').set_index('name', drop=False)
df_moves = pd.read_csv('../data/raw/showdown_moves.csv').set_index('name', drop=False)
type_chart = TeamSynergyScorer().type_chart

def get_moves(*names):
    return [df_moves.loc[n].to_dict() for n in names]

In [3]:
incineroar = BattlePokemon('incineroar', df_pokemon.loc['incineroar'], get_moves('Fake Out', 'Flare Blitz', 'Knock Off'), 'Intimidate')
rillaboom = BattlePokemon('rillaboom', df_pokemon.loc['rillaboom'], get_moves('Wood Hammer', 'Grassy Glide', 'U-turn'), 'Grassy Surge')
flutter_mane = BattlePokemon('flutter-mane', df_pokemon.loc['flutter-mane'], get_moves('Dazzling Gleam', 'Shadow Ball', 'Moonblast'), 'Protosynthesis')
chi_yu = BattlePokemon('chi-yu', df_pokemon.loc['chi-yu'], get_moves('Heat Wave', 'Dark Pulse', 'Overheat'), 'Beads of Ruin')

In [4]:
state = BattleState()
state.team_a_active = [incineroar, rillaboom]
state.team_b_active = [flutter_mane, chi_yu]

engine = TurnEngine(state, type_chart)
agent = SmartVGCAgent(engine)

In [5]:
actions = []
for p in state.team_a_active:
    act = agent.get_best_action(p, state.team_b_active)
    if act: actions.append(act)

for p in state.team_b_active:
    act = agent.get_best_action(p, state.team_a_active)
    if act: actions.append(act)

In [6]:
print("--- AI SELECTED ACTIONS ---")
for a in actions:
    targets = ", ".join([t.name for t in a['targets']])
    print(f"{a['user'].name} chose {a['move']['name']} targeting {targets}")

print("\n--- TRIGGERING ENTRY ABILITIES ---")
engine.trigger_entry_abilities()

print("\n--- RESOLVING TURN ---")
logs = engine.execute_turn(actions)
for log in logs:
    print(log)

--- AI SELECTED ACTIONS ---
incineroar chose Flare Blitz targeting flutter-mane
rillaboom chose Wood Hammer targeting flutter-mane
flutter-mane chose Dazzling Gleam targeting incineroar, rillaboom
chi-yu chose Overheat targeting rillaboom

--- TRIGGERING ENTRY ABILITIES ---
[Ability] incineroar's Intimidate cuts opposing attack!
[Ability] rillaboom's Grassy Surge turned the ground to grass!

--- RESOLVING TURN ---
flutter-mane used Dazzling Gleam! Dealt 52 to incineroar. (150/202 HP)
flutter-mane used Dazzling Gleam! Dealt 57 to rillaboom. (150/207 HP)
chi-yu used Overheat! Dealt 233 to rillaboom. (0/207 HP)
incineroar used Flare Blitz! Dealt 121 to flutter-mane. (41/162 HP)
